# 🌦️ Weather Prediction Model Training

This notebook trains ML models to predict weather conditions using historical data.

**Models trained:**
- XGBoost (best for tabular data)
- LSTM Neural Network (time series)
- Random Forest (baseline)

**Predictions:**
- Temperature (next 24 hours)
- Precipitation probability
- Severe weather classification

---

## 1. Setup & Installation

In [ ]:
# Install required packages
!pip install xgboost lightgbm scikit-learn pandas numpy matplotlib seaborn requests openmeteo-requests requests-cache retry-requests tensorflow keras --quiet

print("✓ Packages installed successfully!")

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Check GPU availability
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")
print("✓ All imports successful!")

## 2. Fetch Historical Weather Data

We'll use the **Open-Meteo Historical API** (free, no API key required)

In [ ]:
# Configuration - EDIT THESE VALUES
CONFIG = {
    # Location (default: New York City)
    'latitude': 40.7128,
    'longitude': -74.0060,
    'location_name': 'New York City',
    
    # Date range (more data = better model, but longer download)
    'start_date': '2019-01-01',
    'end_date': '2024-01-01',
    
    # Timezone
    'timezone': 'America/New_York'
}

print(f"📍 Location: {CONFIG['location_name']}")
print(f"📅 Date range: {CONFIG['start_date']} to {CONFIG['end_date']}")

In [ ]:
def fetch_historical_weather(config):
    """Fetch historical weather data from Open-Meteo API"""
    
    base_url = "https://archive-api.open-meteo.com/v1/archive"
    
    params = {
        "latitude": config['latitude'],
        "longitude": config['longitude'],
        "start_date": config['start_date'],
        "end_date": config['end_date'],
        "timezone": config['timezone'],
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "dew_point_2m",
            "apparent_temperature",
            "pressure_msl",
            "surface_pressure",
            "precipitation",
            "rain",
            "snowfall",
            "cloud_cover",
            "cloud_cover_low",
            "cloud_cover_mid",
            "cloud_cover_high",
            "wind_speed_10m",
            "wind_speed_100m",
            "wind_direction_10m",
            "wind_gusts_10m",
            "soil_temperature_0_to_7cm",
            "soil_moisture_0_to_7cm"
        ]
    }
    
    print("⏳ Fetching data from Open-Meteo (this may take a minute)...")
    response = requests.get(base_url, params=params)
    
    if response.status_code != 200:
        raise Exception(f"API Error: {response.status_code} - {response.text}")
    
    data = response.json()
    
    # Convert to DataFrame
    df = pd.DataFrame(data['hourly'])
    df['time'] = pd.to_datetime(df['time'])
    df.set_index('time', inplace=True)
    
    print(f"✓ Downloaded {len(df):,} hours of weather data")
    print(f"✓ Date range: {df.index.min()} to {df.index.max()}")
    print(f"✓ Features: {len(df.columns)}")
    
    return df

# Fetch the data
df_raw = fetch_historical_weather(CONFIG)
df_raw.head()

In [ ]:
# Data overview
print("\n📊 Data Overview:")
print("=" * 50)
print(df_raw.info())
print("\n📈 Statistics:")
df_raw.describe()

## 3. Data Preprocessing & Feature Engineering

In [ ]:
def preprocess_weather_data(df):
    """Clean and preprocess weather data"""
    
    df = df.copy()
    
    # Handle missing values
    print(f"Missing values before: {df.isnull().sum().sum()}")
    
    # Forward fill then backward fill for time series continuity
    df = df.fillna(method='ffill').fillna(method='bfill')
    
    print(f"Missing values after: {df.isnull().sum().sum()}")
    
    return df

df_clean = preprocess_weather_data(df_raw)
print("✓ Data cleaned")

In [ ]:
def create_features(df):
    """Create time-based and lag features for ML"""
    
    df = df.copy()
    
    # Time-based features
    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek
    df['day_of_year'] = df.index.dayofyear
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)
    
    # Cyclical encoding for time features (captures periodicity)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)
    
    # Lag features (previous hours' data)
    lag_features = ['temperature_2m', 'pressure_msl', 'wind_speed_10m', 'precipitation', 'relative_humidity_2m']
    lag_hours = [1, 3, 6, 12, 24]
    
    for feature in lag_features:
        for lag in lag_hours:
            df[f'{feature}_lag_{lag}h'] = df[feature].shift(lag)
    
    # Rolling statistics (moving averages, std)
    rolling_windows = [6, 12, 24]
    
    for feature in ['temperature_2m', 'pressure_msl', 'wind_speed_10m']:
        for window in rolling_windows:
            df[f'{feature}_rolling_mean_{window}h'] = df[feature].rolling(window=window).mean()
            df[f'{feature}_rolling_std_{window}h'] = df[feature].rolling(window=window).std()
    
    # Pressure change (important for weather prediction)
    df['pressure_change_3h'] = df['pressure_msl'] - df['pressure_msl'].shift(3)
    df['pressure_change_6h'] = df['pressure_msl'] - df['pressure_msl'].shift(6)
    df['pressure_change_24h'] = df['pressure_msl'] - df['pressure_msl'].shift(24)
    
    # Temperature change
    df['temp_change_3h'] = df['temperature_2m'] - df['temperature_2m'].shift(3)
    df['temp_change_24h'] = df['temperature_2m'] - df['temperature_2m'].shift(24)
    
    # Wind direction components (convert degrees to x,y)
    df['wind_dir_sin'] = np.sin(np.radians(df['wind_direction_10m']))
    df['wind_dir_cos'] = np.cos(np.radians(df['wind_direction_10m']))
    
    # Dew point depression (indicator of precipitation likelihood)
    df['dew_point_depression'] = df['temperature_2m'] - df['dew_point_2m']
    
    # Drop rows with NaN from lag/rolling features
    df = df.dropna()
    
    print(f"✓ Created {len(df.columns)} features")
    print(f"✓ Dataset size: {len(df):,} samples")
    
    return df

df_features = create_features(df_clean)
df_features.head()

In [ ]:
# Create target variables
def create_targets(df, forecast_hours=24):
    """Create prediction targets"""
    
    df = df.copy()
    
    # Target 1: Temperature in X hours
    df['target_temp_24h'] = df['temperature_2m'].shift(-24)
    df['target_temp_12h'] = df['temperature_2m'].shift(-12)
    df['target_temp_6h'] = df['temperature_2m'].shift(-6)
    
    # Target 2: Will it rain in next 24 hours? (binary classification)
    df['target_rain_24h'] = (df['precipitation'].shift(-1).rolling(24).sum().shift(-23) > 0.1).astype(int)
    
    # Target 3: Precipitation amount in next 24h
    df['target_precip_24h'] = df['precipitation'].shift(-1).rolling(24).sum().shift(-23)
    
    # Target 4: Severe weather (high wind or heavy precip)
    future_wind = df['wind_gusts_10m'].shift(-1).rolling(24).max().shift(-23)
    future_precip = df['precipitation'].shift(-1).rolling(24).sum().shift(-23)
    df['target_severe_24h'] = ((future_wind > 50) | (future_precip > 25)).astype(int)
    
    # Drop rows where targets are NaN
    df = df.dropna()
    
    print(f"✓ Created target variables")
    print(f"✓ Final dataset size: {len(df):,} samples")
    
    return df

df_final = create_targets(df_features)
print(f"\nTarget distribution:")
print(f"  Rain in 24h: {df_final['target_rain_24h'].mean()*100:.1f}% positive")
print(f"  Severe weather: {df_final['target_severe_24h'].mean()*100:.1f}% positive")

## 4. Data Visualization

In [ ]:
# Visualize the data
fig, axes = plt.subplots(3, 2, figsize=(15, 12))
fig.suptitle(f'Weather Data Overview - {CONFIG["location_name"]}', fontsize=14)

# Temperature over time
axes[0, 0].plot(df_final.index, df_final['temperature_2m'], alpha=0.7, linewidth=0.5)
axes[0, 0].set_title('Temperature Over Time')
axes[0, 0].set_ylabel('Temperature (°C)')

# Temperature distribution
axes[0, 1].hist(df_final['temperature_2m'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Temperature Distribution')
axes[0, 1].set_xlabel('Temperature (°C)')

# Precipitation
monthly_precip = df_final['precipitation'].resample('M').sum()
axes[1, 0].bar(monthly_precip.index, monthly_precip.values, width=20, alpha=0.7)
axes[1, 0].set_title('Monthly Precipitation')
axes[1, 0].set_ylabel('Precipitation (mm)')

# Wind speed distribution
axes[1, 1].hist(df_final['wind_speed_10m'], bins=50, edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Wind Speed Distribution')
axes[1, 1].set_xlabel('Wind Speed (km/h)')

# Pressure over time
axes[2, 0].plot(df_final.index, df_final['pressure_msl'], alpha=0.7, linewidth=0.5)
axes[2, 0].set_title('Pressure Over Time')
axes[2, 0].set_ylabel('Pressure (hPa)')

# Average temperature by hour
hourly_temp = df_final.groupby('hour')['temperature_2m'].mean()
axes[2, 1].plot(hourly_temp.index, hourly_temp.values, marker='o')
axes[2, 1].set_title('Average Temperature by Hour')
axes[2, 1].set_xlabel('Hour of Day')
axes[2, 1].set_ylabel('Temperature (°C)')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for key features
key_features = ['temperature_2m', 'relative_humidity_2m', 'pressure_msl', 'wind_speed_10m', 
                'precipitation', 'cloud_cover', 'dew_point_2m', 'target_temp_24h']

plt.figure(figsize=(10, 8))
correlation = df_final[key_features].corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Prepare Data for Training

In [ ]:
# Define feature columns (exclude targets and raw time columns)
target_columns = ['target_temp_24h', 'target_temp_12h', 'target_temp_6h', 
                  'target_rain_24h', 'target_precip_24h', 'target_severe_24h']

exclude_columns = target_columns + ['year']  # Exclude year to prevent overfitting to specific years

feature_columns = [col for col in df_final.columns if col not in exclude_columns]

print(f"Number of features: {len(feature_columns)}")
print(f"\nFeatures: {feature_columns[:10]}...")

In [ ]:
# Prepare datasets for different prediction tasks
X = df_final[feature_columns]

# Task 1: Temperature prediction (regression)
y_temp = df_final['target_temp_24h']

# Task 2: Rain prediction (classification)
y_rain = df_final['target_rain_24h']

# Task 3: Severe weather prediction (classification)
y_severe = df_final['target_severe_24h']

# Time-based train/test split (important for time series!)
# Use last 20% of data for testing
split_idx = int(len(X) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_temp_train, y_temp_test = y_temp.iloc[:split_idx], y_temp.iloc[split_idx:]
y_rain_train, y_rain_test = y_rain.iloc[:split_idx], y_rain.iloc[split_idx:]
y_severe_train, y_severe_test = y_severe.iloc[:split_idx], y_severe.iloc[split_idx:]

print(f"Training samples: {len(X_train):,}")
print(f"Testing samples: {len(X_test):,}")
print(f"\nTraining period: {X_train.index.min()} to {X_train.index.max()}")
print(f"Testing period: {X_test.index.min()} to {X_test.index.max()}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ Features scaled")

## 6. Train Models

### 6.1 Temperature Prediction (Regression)

In [ ]:
# XGBoost for temperature prediction
print("Training XGBoost for temperature prediction...")

xgb_temp = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50
)

xgb_temp.fit(
    X_train_scaled, y_temp_train,
    eval_set=[(X_test_scaled, y_temp_test)],
    verbose=False
)

# Predictions
y_temp_pred_xgb = xgb_temp.predict(X_test_scaled)

# Metrics
mae_xgb = mean_absolute_error(y_temp_test, y_temp_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_temp_test, y_temp_pred_xgb))
r2_xgb = r2_score(y_temp_test, y_temp_pred_xgb)

print(f"\n✓ XGBoost Temperature Results:")
print(f"  MAE: {mae_xgb:.2f}°C")
print(f"  RMSE: {rmse_xgb:.2f}°C")
print(f"  R² Score: {r2_xgb:.4f}")

In [ ]:
# LightGBM for temperature prediction
print("Training LightGBM for temperature prediction...")

lgb_temp = lgb.LGBMRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgb_temp.fit(
    X_train_scaled, y_temp_train,
    eval_set=[(X_test_scaled, y_temp_test)]
)

# Predictions
y_temp_pred_lgb = lgb_temp.predict(X_test_scaled)

# Metrics
mae_lgb = mean_absolute_error(y_temp_test, y_temp_pred_lgb)
rmse_lgb = np.sqrt(mean_squared_error(y_temp_test, y_temp_pred_lgb))
r2_lgb = r2_score(y_temp_test, y_temp_pred_lgb)

print(f"\n✓ LightGBM Temperature Results:")
print(f"  MAE: {mae_lgb:.2f}°C")
print(f"  RMSE: {rmse_lgb:.2f}°C")
print(f"  R² Score: {r2_lgb:.4f}")

In [ ]:
# Random Forest baseline
print("Training Random Forest for temperature prediction...")

rf_temp = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf_temp.fit(X_train_scaled, y_temp_train)

# Predictions
y_temp_pred_rf = rf_temp.predict(X_test_scaled)

# Metrics
mae_rf = mean_absolute_error(y_temp_test, y_temp_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_temp_test, y_temp_pred_rf))
r2_rf = r2_score(y_temp_test, y_temp_pred_rf)

print(f"\n✓ Random Forest Temperature Results:")
print(f"  MAE: {mae_rf:.2f}°C")
print(f"  RMSE: {rmse_rf:.2f}°C")
print(f"  R² Score: {r2_rf:.4f}")

### 6.2 LSTM Neural Network for Temperature

In [ ]:
# Prepare sequence data for LSTM
def create_sequences(X, y, seq_length=24):
    """Create sequences for LSTM input"""
    Xs, ys = [], []
    for i in range(len(X) - seq_length):
        Xs.append(X[i:(i + seq_length)])
        ys.append(y.iloc[i + seq_length])
    return np.array(Xs), np.array(ys)

SEQ_LENGTH = 24  # Use last 24 hours to predict

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_temp_train, SEQ_LENGTH)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_temp_test, SEQ_LENGTH)

print(f"LSTM input shape: {X_train_seq.shape}")
print(f"  - Samples: {X_train_seq.shape[0]}")
print(f"  - Time steps: {X_train_seq.shape[1]}")
print(f"  - Features: {X_train_seq.shape[2]}")

In [ ]:
# Build LSTM model
print("Building LSTM model...")

lstm_model = Sequential([
    Bidirectional(LSTM(64, return_sequences=True), input_shape=(SEQ_LENGTH, X_train_seq.shape[2])),
    Dropout(0.2),
    Bidirectional(LSTM(32, return_sequences=False)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1)
])

lstm_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

lstm_model.summary()

In [ ]:
# Train LSTM
print("\nTraining LSTM (this may take a few minutes)...")

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

history = lstm_model.fit(
    X_train_seq, y_train_seq,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# LSTM predictions
y_temp_pred_lstm = lstm_model.predict(X_test_seq, verbose=0).flatten()

mae_lstm = mean_absolute_error(y_test_seq, y_temp_pred_lstm)
rmse_lstm = np.sqrt(mean_squared_error(y_test_seq, y_temp_pred_lstm))
r2_lstm = r2_score(y_test_seq, y_temp_pred_lstm)

print(f"\n✓ LSTM Temperature Results:")
print(f"  MAE: {mae_lstm:.2f}°C")
print(f"  RMSE: {rmse_lstm:.2f}°C")
print(f"  R² Score: {r2_lstm:.4f}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('LSTM Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].legend()

axes[1].plot(history.history['mae'], label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_title('LSTM Training MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE (°C)')
axes[1].legend()

plt.tight_layout()
plt.show()

### 6.3 Rain Prediction (Classification)

In [ ]:
# XGBoost for rain classification
print("Training XGBoost for rain prediction...")

xgb_rain = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=len(y_rain_train[y_rain_train==0]) / len(y_rain_train[y_rain_train==1]),  # Handle imbalance
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=30,
    eval_metric='logloss'
)

xgb_rain.fit(
    X_train_scaled, y_rain_train,
    eval_set=[(X_test_scaled, y_rain_test)],
    verbose=False
)

# Predictions
y_rain_pred = xgb_rain.predict(X_test_scaled)
y_rain_prob = xgb_rain.predict_proba(X_test_scaled)[:, 1]

print(f"\n✓ XGBoost Rain Prediction Results:")
print(classification_report(y_rain_test, y_rain_pred, target_names=['No Rain', 'Rain']))

In [ ]:
# Confusion matrix for rain prediction
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_rain_test, y_rain_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Rain', 'Rain'], 
            yticklabels=['No Rain', 'Rain'])
plt.title('Rain Prediction Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

### 6.4 Severe Weather Prediction

In [ ]:
# XGBoost for severe weather classification
print("Training XGBoost for severe weather prediction...")

# Calculate class weight for imbalanced data
pos_weight = len(y_severe_train[y_severe_train==0]) / max(len(y_severe_train[y_severe_train==1]), 1)

xgb_severe = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

xgb_severe.fit(X_train_scaled, y_severe_train, verbose=False)

# Predictions
y_severe_pred = xgb_severe.predict(X_test_scaled)
y_severe_prob = xgb_severe.predict_proba(X_test_scaled)[:, 1]

print(f"\n✓ XGBoost Severe Weather Results:")
print(classification_report(y_severe_test, y_severe_pred, target_names=['Normal', 'Severe']))

## 7. Model Comparison & Feature Importance

In [ ]:
# Compare all temperature models
results = pd.DataFrame({
    'Model': ['XGBoost', 'LightGBM', 'Random Forest', 'LSTM'],
    'MAE (°C)': [mae_xgb, mae_lgb, mae_rf, mae_lstm],
    'RMSE (°C)': [rmse_xgb, rmse_lgb, rmse_rf, rmse_lstm],
    'R² Score': [r2_xgb, r2_lgb, r2_rf, r2_lstm]
})

print("\n" + "="*60)
print("TEMPERATURE PREDICTION MODEL COMPARISON")
print("="*60)
print(results.to_string(index=False))
print("\nBest model:", results.loc[results['MAE (°C)'].idxmin(), 'Model'])

In [ ]:
# Feature importance (XGBoost)
feature_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': xgb_temp.feature_importances_
}).sort_values('Importance', ascending=False)

# Plot top 20 features
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(20)
plt.barh(range(len(top_features)), top_features['Importance'].values)
plt.yticks(range(len(top_features)), top_features['Feature'].values)
plt.xlabel('Feature Importance')
plt.title('Top 20 Most Important Features for Temperature Prediction')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 features:")
print(feature_importance.head(10).to_string(index=False))

In [ ]:
# Visualize predictions vs actual
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sample 500 points for clarity
sample_size = min(500, len(y_temp_test))
sample_idx = np.random.choice(len(y_temp_test), sample_size, replace=False)
sample_idx = np.sort(sample_idx)

# XGBoost predictions
axes[0, 0].scatter(y_temp_test.values[sample_idx], y_temp_pred_xgb[sample_idx], alpha=0.5, s=10)
axes[0, 0].plot([y_temp_test.min(), y_temp_test.max()], [y_temp_test.min(), y_temp_test.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual Temperature (°C)')
axes[0, 0].set_ylabel('Predicted Temperature (°C)')
axes[0, 0].set_title(f'XGBoost (MAE: {mae_xgb:.2f}°C)')

# LightGBM predictions
axes[0, 1].scatter(y_temp_test.values[sample_idx], y_temp_pred_lgb[sample_idx], alpha=0.5, s=10)
axes[0, 1].plot([y_temp_test.min(), y_temp_test.max()], [y_temp_test.min(), y_temp_test.max()], 'r--', lw=2)
axes[0, 1].set_xlabel('Actual Temperature (°C)')
axes[0, 1].set_ylabel('Predicted Temperature (°C)')
axes[0, 1].set_title(f'LightGBM (MAE: {mae_lgb:.2f}°C)')

# Time series comparison (last 200 hours)
plot_range = 200
axes[1, 0].plot(range(plot_range), y_temp_test.values[-plot_range:], label='Actual', alpha=0.8)
axes[1, 0].plot(range(plot_range), y_temp_pred_xgb[-plot_range:], label='XGBoost', alpha=0.8)
axes[1, 0].set_xlabel('Hours')
axes[1, 0].set_ylabel('Temperature (°C)')
axes[1, 0].set_title('Temperature Forecast vs Actual (Last 200 Hours)')
axes[1, 0].legend()

# Prediction error distribution
errors = y_temp_test.values - y_temp_pred_xgb
axes[1, 1].hist(errors, bins=50, edgecolor='black', alpha=0.7)
axes[1, 1].axvline(x=0, color='r', linestyle='--', lw=2)
axes[1, 1].set_xlabel('Prediction Error (°C)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title(f'XGBoost Error Distribution (Mean: {errors.mean():.2f}°C)')

plt.tight_layout()
plt.show()

## 8. Save Models for Deployment

In [ ]:
import pickle
import json

# Create models directory
import os
os.makedirs('weather_models', exist_ok=True)

# Save XGBoost models
xgb_temp.save_model('weather_models/xgb_temperature.json')
xgb_rain.save_model('weather_models/xgb_rain.json')
xgb_severe.save_model('weather_models/xgb_severe.json')

# Save LightGBM model
lgb_temp.booster_.save_model('weather_models/lgb_temperature.txt')

# Save LSTM model
lstm_model.save('weather_models/lstm_temperature.keras')

# Save scaler
with open('weather_models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save feature columns
with open('weather_models/feature_columns.json', 'w') as f:
    json.dump(feature_columns, f)

# Save model config
model_config = {
    'location': CONFIG,
    'feature_count': len(feature_columns),
    'lstm_seq_length': SEQ_LENGTH,
    'models': {
        'temperature': {
            'xgboost': {'mae': mae_xgb, 'rmse': rmse_xgb, 'r2': r2_xgb},
            'lightgbm': {'mae': mae_lgb, 'rmse': rmse_lgb, 'r2': r2_lgb},
            'lstm': {'mae': mae_lstm, 'rmse': rmse_lstm, 'r2': r2_lstm}
        }
    },
    'training_date': datetime.now().isoformat()
}

with open('weather_models/model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

print("✓ All models saved to 'weather_models/' directory")
print("\nSaved files:")
for f in os.listdir('weather_models'):
    size = os.path.getsize(f'weather_models/{f}') / 1024
    print(f"  - {f} ({size:.1f} KB)")

In [ ]:
# Download models to Google Drive (optional)
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Copy models to Drive
!cp -r weather_models /content/drive/MyDrive/

print("✓ Models copied to Google Drive!")

## 9. Example: Making Predictions with Saved Models

In [ ]:
# Example of loading and using saved models
def load_models():
    """Load all saved models"""
    models = {}
    
    # Load XGBoost
    models['xgb_temp'] = xgb.XGBRegressor()
    models['xgb_temp'].load_model('weather_models/xgb_temperature.json')
    
    models['xgb_rain'] = xgb.XGBClassifier()
    models['xgb_rain'].load_model('weather_models/xgb_rain.json')
    
    models['xgb_severe'] = xgb.XGBClassifier()
    models['xgb_severe'].load_model('weather_models/xgb_severe.json')
    
    # Load scaler
    with open('weather_models/scaler.pkl', 'rb') as f:
        models['scaler'] = pickle.load(f)
    
    # Load feature columns
    with open('weather_models/feature_columns.json', 'r') as f:
        models['features'] = json.load(f)
    
    return models

def predict_weather(models, current_data):
    """
    Make predictions using loaded models
    
    Args:
        models: dict of loaded models
        current_data: DataFrame with current weather features
    
    Returns:
        dict with predictions
    """
    # Scale features
    X = models['scaler'].transform(current_data[models['features']])
    
    # Make predictions
    predictions = {
        'temperature_24h': models['xgb_temp'].predict(X)[0],
        'rain_probability': models['xgb_rain'].predict_proba(X)[0, 1],
        'severe_probability': models['xgb_severe'].predict_proba(X)[0, 1]
    }
    
    return predictions

# Test with last row of data
loaded_models = load_models()
test_prediction = predict_weather(loaded_models, df_final.iloc[[-1]])

print("\n" + "="*50)
print("SAMPLE PREDICTION")
print("="*50)
print(f"Temperature in 24 hours: {test_prediction['temperature_24h']:.1f}°C")
print(f"Rain probability (24h): {test_prediction['rain_probability']*100:.1f}%")
print(f"Severe weather probability (24h): {test_prediction['severe_probability']*100:.1f}%")

## 10. Export for Web App (JavaScript Compatible)

To use these models in your web app, you can:
1. Use ONNX format for browser-based inference
2. Create a simple Python API server
3. Use TensorFlow.js for the LSTM model

In [ ]:
# Convert LSTM to TensorFlow.js format
!pip install tensorflowjs --quiet

import tensorflowjs as tfjs

# Save for TensorFlow.js
tfjs.converters.save_keras_model(lstm_model, 'weather_models/tfjs_lstm')

print("✓ LSTM model exported for TensorFlow.js")
print("\nTo use in browser:")
print("  const model = await tf.loadLayersModel('tfjs_lstm/model.json');")

In [ ]:
# Convert XGBoost to ONNX for browser use
!pip install onnx skl2onnx onnxmltools onnxruntime --quiet

from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType

# Define input shape
initial_types = [('features', FloatTensorType([None, len(feature_columns)]))]

# Convert XGBoost temperature model to ONNX
onnx_model = convert_xgboost(xgb_temp, initial_types=initial_types, target_opset=12)

# Save ONNX model
with open('weather_models/xgb_temperature.onnx', 'wb') as f:
    f.write(onnx_model.SerializeToString())

print("✓ XGBoost model exported to ONNX format")
print("\nTo use in browser with ONNX.js:")
print("  const session = await ort.InferenceSession.create('xgb_temperature.onnx');")

## 11. Summary & Next Steps

### What we built:
- **Temperature prediction model** (XGBoost) - Predicts temperature 24 hours ahead
- **Rain prediction model** (XGBoost) - Binary classification for rain in next 24h
- **Severe weather model** (XGBoost) - Predicts high wind/heavy precipitation events
- **LSTM model** - Deep learning alternative for temperature forecasting

### Model files saved:
- `xgb_temperature.json` - XGBoost temperature model
- `xgb_rain.json` - XGBoost rain classifier
- `xgb_severe.json` - XGBoost severe weather classifier
- `lstm_temperature.keras` - LSTM model
- `scaler.pkl` - Feature scaler
- `feature_columns.json` - List of required features
- `tfjs_lstm/` - TensorFlow.js compatible LSTM
- `xgb_temperature.onnx` - ONNX format for browser

### To improve accuracy:
1. Add more years of historical data
2. Include data from nearby weather stations
3. Add external features (solar radiation, elevation, etc.)
4. Ensemble multiple models
5. Train location-specific models

### Integration options:
1. **Python backend** - Load models directly with pickle/xgboost
2. **Browser inference** - Use ONNX.js or TensorFlow.js
3. **API service** - Create Flask/FastAPI endpoint

In [ ]:
# Final summary
print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"\nLocation: {CONFIG['location_name']}")
print(f"Training data: {CONFIG['start_date']} to {CONFIG['end_date']}")
print(f"Total samples: {len(df_final):,}")
print(f"Features used: {len(feature_columns)}")
print(f"\nBest temperature model: XGBoost (MAE: {mae_xgb:.2f}°C)")
print(f"\nModels saved to: weather_models/")
print("\nReady for deployment!")